# Customer Churn Analysis for Telecom Industry

---



In [3]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
!pip install eli5
import eli5
from eli5.sklearn import PermutationImportance
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and SQL Aggregation



In [4]:
# Load the dataset
df = pd.read_csv('telecom_customer_data.csv')

# in-memory SQLite database to run our SQL queries
conn = sqlite3.connect(':memory:')
df.to_sql('customers', conn, index=False)

# SQL aggregation: call duration, complaints, and recharge frequency by Plan Type
query = """
SELECT
    UPPER(Plan_Type) as Plan_Type,
    COUNT(*) as Total_Customers,
    AVG(Call_Duration_Mins) as Avg_Call_Duration,
    SUM(Num_Complaints) as Total_Complaints,
    AVG(Recharge_Frequency) as Avg_Recharge
FROM customers
GROUP BY UPPER(Plan_Type)
"""
agg_df = pd.read_sql(query, conn)
print("Here is our SQL Aggregation Result:")
display(agg_df)

Here is our SQL Aggregation Result:


,Plan_Type,Total_Customers,Avg_Call_Duration,Total_Complaints,Avg_Recharge
0,POSTPAID,25692,287.221377,38800,3.187763
1,PREPAID,25808,286.159289,38884,3.208811


## 2. Data Cleaning


In [5]:
# Fix negative call durations and data usage
df['Call_Duration_Mins'] = df['Call_Duration_Mins'].apply(lambda x: x if x >= 0 else 0)
df['Data_Usage_GB'] = df['Data_Usage_GB'].apply(lambda x: x if x >= 0 else 0)

# Fill missing values with median for numeric columns
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Data_Usage_GB'].fillna(df['Data_Usage_GB'].median(), inplace=True)

# Standardize gender column
df['Gender'] = df['Gender'].str.upper().str[0]
df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female', 'O': 'Other'})
df['Gender'].fillna('Unknown', inplace=True)

# Standardize plan type
df['Plan_Type'] = df['Plan_Type'].str.upper()

# Drop duplicates based on CustomerID
df = df.drop_duplicates(subset=['CustomerID'])

print(f"Data cleaned! We now have {len(df)} unique records.")

Data cleaned! We now have 50000 unique records.


## 3. Feature Engineering and Preprocessing


In [6]:
# One-hot encode categorical features
df_model = pd.get_dummies(df.drop('CustomerID', axis=1), columns=['Gender', 'Plan_Type'], drop_first=True)

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training data shape:", X_train.shape)

Training data shape: (40000, 10)


## 4. Modeling: Random Forest


In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=8)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

Model Accuracy: 0.8535

Classification Report:
               precision    recall  f1-score   support

           0       0.85      1.00      0.92      8543
           1       0.32      0.00      0.01      1457

    accuracy                           0.85     10000
   macro avg       0.59      0.50      0.47     10000
weighted avg       0.78      0.85      0.79     10000



## 5. Model Explainability with ELI5


In [8]:
# Calculate permutation importance for better interpretability
perm = PermutationImportance(rf, random_state=42).fit(X_test, y_test)
eli5.show_weights(perm, feature_names = X_test.columns.tolist())

Weight,Feature
0.0004 ± 0.0008,Num_Complaints
0.0001 ± 0.0003,Recharge_Frequency
0.0000 ± 0.0003,Plan_Type_PREPAID
-0.0000 ± 0.0007,Customer_Service_Calls
-0.0000 ± 0.0001,Gender_Unknown
-0.0001 ± 0.0002,Gender_Other
-0.0002 ± 0.0003,Gender_Male
-0.0005 ± 0.0005,Age
-0.0005 ± 0.0008,Call_Duration_Mins
-0.0006 ± 0.0007,Data_Usage_GB


## 6. Customer Segmentation


In [9]:
df['Churn_Probability'] = rf.predict_proba(X)[:, 1]

def segment_customer(row):
    if row['Churn_Probability'] > 0.6:
        return 'At Risk'
    elif row['Call_Duration_Mins'] < 10 and row['Data_Usage_GB'] < 1:
        return 'Dormant'
    else:
        return 'Loyal'

df['Segment'] = df.apply(segment_customer, axis=1)
print(df['Segment'].value_counts())

# Save the segmented data for business users
df.to_csv('segmented_customers.csv', index=False)
print("\nCustomer segmentation complete and saved to 'segmented_customers.csv'!")

Segment
Loyal      49681
Dormant      281
At Risk       38
Name: count, dtype: int64

Customer segmentation complete and saved to 'segmented_customers.csv'!
